# Ekstraksi Soal & Materi dari PDF/Ebook dengan VLM (Gemini)

Notebook ini membaca **setiap halaman PDF sebagai gambar** lalu memakai VLM
(Gemini Flash) untuk:

1. Memisahkan **`soal`** (problem) vs **`materi`** (teori/definisi/penjelasan).
2. Untuk soal: memisah **`question` / `answer` / `steps`** (cara).
3. Membuang noise: header/footer, nomor halaman, watermark, URL, nama penulis.
4. Mempertahankan notasi matematika sebagai **LaTeX** inline `$...$`.

Kenapa VLM (baca gambar) bukan `pdfplumber`? Ekstraksi teks merusak notasi math
(`(cid:18)`, pecahan kebalik) dan mencampur soal+jawaban. VLM yang melihat
halaman menangani ini jauh lebih baik.

**Alur pemakaian:** isi API key → jalankan sel prototype di beberapa PDF →
cek hasil → jalankan batch penuh.

> Backend di-abstraksi lewat `call_vlm()`. Mau pindah ke Qwen2.5-VL lokal/Kaggle?
> Cukup ganti satu fungsi; prompt & schema tetap sama.


## 1. Install dependencies

In [ ]:
# Jalankan sekali. Aman diulang.
%pip install -q google-genai pymupdf pillow tqdm python-dotenv


## 2. Konfigurasi

API key dibaca dari environment / file `.env` di root proyek
(`GEMINI_API_KEY=...`). **Jangan** hard-code key di notebook (`.env` sudah masuk
`.gitignore`).

Dapatkan key gratis di Google AI Studio.

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()  # baca .env di cwd / root

# ── Path ──────────────────────────────────────────────────────────────
# Jalankan notebook dari root proyek, atau set PROJECT_ROOT manual.
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
OUT_DIR = PROJECT_ROOT / "data" / "extracted_vlm"      # tidak menimpa hasil pdfplumber lama
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Model & backend ───────────────────────────────────────────────────
BACKEND   = "gemini"                 # "gemini" | "qwen" (lihat sel backend)
MODEL_ID  = "gemini-2.0-flash"       # bisa diganti ke model Flash terbaru
DPI       = 150                      # naikkan ke 200 utk halaman padat (lebih akurat, lebih mahal)
TEMPERATURE = 0.0

# ── Rate limit / retry (free tier) ────────────────────────────────────
SLEEP_BETWEEN = 1.0                  # detik antar request (hindari kena RPM limit)
MAX_RETRIES   = 4

# ── Resume ────────────────────────────────────────────────────────────
OVERWRITE = False                    # True = proses ulang PDF yg outputnya sudah ada

API_KEY = os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
assert API_KEY, "Set GEMINI_API_KEY di .env atau environment dulu."

print("PROJECT_ROOT :", PROJECT_ROOT)
print("RAW_DIR      :", RAW_DIR, "(ada)" if RAW_DIR.exists() else "(TIDAK ADA)")
print("OUT_DIR      :", OUT_DIR)
print("Backend/model:", BACKEND, "/", MODEL_ID)


## 3. Prompt ekstraksi

Satu prompt berbahasa Indonesia yang memberi tahu VLM cara memisahkan soal vs
materi dan membuang noise. Output dipaksa JSON array.

In [ ]:
EXTRACTION_PROMPT = """Kamu adalah ekstraktor data dari SATU halaman buku/kumpulan soal matematika berbahasa Indonesia.
Aku beri satu gambar halaman. Keluarkan HANYA sebuah JSON array (tanpa teks lain, tanpa pagar ```).

Setiap elemen array adalah salah satu dari dua tipe:

1) SOAL — sesuatu yang harus diselesaikan (soal latihan, contoh soal, soal olimpiade, pertanyaan):
{
  "type": "soal",
  "nomor": "<nomor soal jika ada, mis. '1', '12a'; jika tidak ada kosongkan ''>",
  "question": "<teks soal lengkap; tulis notasi matematika sebagai LaTeX inline $...$>",
  "answer": "<jawaban akhir jika tertera di halaman; jika tidak ada ''>",
  "steps": "<langkah penyelesaian / pembahasan / cara jika ada di halaman; jika tidak ada ''>",
  "has_image": <true jika soal bergantung pada gambar/diagram/grafik, selain itu false>
}

2) MATERI — teori, definisi, teorema, sifat, rumus, penjelasan (BUKAN soal):
{
  "type": "materi",
  "judul": "<judul/subbab jika ada; jika tidak ada ''>",
  "content": "<isi materi; notasi matematika sebagai LaTeX inline $...$>"
}

ATURAN PENTING:
- ABAIKAN noise: header/footer, nomor halaman, watermark, nama penulis, URL situs, teks "Halaman X dari Y".
- Pisahkan tiap soal menjadi elemen array tersendiri (ikuti penomoran di halaman).
- "Contoh soal" yang disertai penyelesaian → type "soal": isi question + steps (+ answer bila ada).
- Jika sebuah soal butuh gambar, set has_image=true DAN sisipkan deskripsi singkat diagram di dalam question dengan format "[Gambar: ...]".
- Pertahankan notasi matematika seakurat mungkin sebagai LaTeX. JANGAN mengarang soal/jawaban yang tidak ada di halaman.
- Jika satu soal jelas bersambung dari halaman sebelumnya / terpotong, tetap keluarkan apa adanya.
- Jika halaman tidak memuat soal maupun materi bermakna (sampul, daftar isi, halaman kosong, hanya gambar dekoratif), keluarkan array kosong [].

Keluarkan HANYA JSON array yang valid."""


## 4. Backend VLM (`call_vlm`)

Seam tunggal. Default Gemini. Stub Qwen disertakan supaya swap = ganti `BACKEND`
tanpa menyentuh sisa pipeline.

In [ ]:
from google import genai
from google.genai import types as genai_types

_gemini_client = None

def _gemini_call(pil_image, prompt: str) -> str:
    global _gemini_client
    if _gemini_client is None:
        _gemini_client = genai.Client(api_key=API_KEY)
    resp = _gemini_client.models.generate_content(
        model=MODEL_ID,
        contents=[prompt, pil_image],
        config=genai_types.GenerateContentConfig(
            temperature=TEMPERATURE,
            response_mime_type="application/json",
        ),
    )
    return resp.text or ""


def _qwen_call(pil_image, prompt: str) -> str:
    # ── STUB swap-point untuk Qwen2.5-VL (lokal 4060 / Kaggle / vLLM) ──
    # Implementasi tipikal (transformers):
    #   from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
    #   messages = [{"role":"user","content":[
    #       {"type":"image","image": pil_image},
    #       {"type":"text","text": prompt}]}]
    #   text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    #   inputs = processor(text=[text], images=[pil_image], return_tensors="pt").to(model.device)
    #   out = model.generate(**inputs, max_new_tokens=2048, do_sample=False)
    #   return processor.batch_decode(out[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)[0]
    raise NotImplementedError("Isi _qwen_call untuk pakai backend lokal/Kaggle.")


def call_vlm(pil_image, prompt: str) -> str:
    if BACKEND == "gemini":
        return _gemini_call(pil_image, prompt)
    if BACKEND == "qwen":
        return _qwen_call(pil_image, prompt)
    raise ValueError(f"BACKEND tidak dikenal: {BACKEND}")


## 5. Render halaman & parsing JSON

`PyMuPDF` me-render tiap halaman jadi gambar (tanpa dependency sistem seperti
poppler). Parser JSON dibuat tahan terhadap pagar ```json dan teks liar.

In [ ]:
import io
import json
import re
import time
import fitz  # PyMuPDF
from PIL import Image


def page_to_image(page, dpi: int = DPI) -> Image.Image:
    pix = page.get_pixmap(dpi=dpi)
    return Image.open(io.BytesIO(pix.tobytes("png"))).convert("RGB")


_FENCE_RE = re.compile(r"```(?:json)?\s*(.*?)\s*```", re.DOTALL)

def parse_json_array(text: str):
    """Robust: kembalikan list[dict]. [] kalau gagal."""
    if not text:
        return []
    t = text.strip()
    m = _FENCE_RE.search(t)
    if m:
        t = m.group(1).strip()
    # ambil dari kurung siku pertama sampai terakhir
    i, j = t.find("["), t.rfind("]")
    if i != -1 and j != -1 and j > i:
        t = t[i : j + 1]
    try:
        data = json.loads(t)
    except json.JSONDecodeError:
        return []
    if isinstance(data, dict):
        data = [data]
    return [d for d in data if isinstance(d, dict)]


def call_vlm_with_retry(pil_image, prompt: str) -> str:
    last = None
    for attempt in range(MAX_RETRIES):
        try:
            out = call_vlm(pil_image, prompt)
            if SLEEP_BETWEEN:
                time.sleep(SLEEP_BETWEEN)
            return out
        except Exception as e:  # rate limit / transient
            last = e
            wait = (2 ** attempt) * max(SLEEP_BETWEEN, 1.0)
            print(f"    retry {attempt+1}/{MAX_RETRIES} after error: {e} (sleep {wait:.0f}s)")
            time.sleep(wait)
    print(f"    GAGAL setelah {MAX_RETRIES} percobaan: {last}")
    return ""


## 6. Bangun record & proses per-PDF

Skema soal selaras dengan pipeline yang ada (`question/answer/steps/source_file/
source_page/question_id/has_image`) + field `type` untuk memisahkan soal vs
materi. Tiap PDF → satu file `.jsonl`. Bisa di-resume (skip yang sudah ada).

In [ ]:
def _subject_for(pdf_path: Path) -> str:
    """Ambil 'subject' dari source.json di folder yang sama, jika ada."""
    sj = pdf_path.parent / "source.json"
    if sj.exists():
        try:
            return json.loads(sj.read_text(encoding="utf-8")).get("subject", "")
        except Exception:
            return ""
    return ""


def page_items_to_records(items, pdf_path, page_num, subject):
    records = []
    n_soal = n_materi = 0
    for it in items:
        typ = (it.get("type") or "").lower()
        if typ == "soal":
            n_soal += 1
            records.append({
                "type": "soal",
                "question": (it.get("question") or "").strip(),
                "answer":   (it.get("answer") or "").strip(),
                "steps":    (it.get("steps") or "").strip(),
                "nomor":    str(it.get("nomor") or "").strip(),
                "has_image": bool(it.get("has_image", False)),
                "source_file":   pdf_path.name,
                "source_folder": pdf_path.parent.name,
                "subject":       subject,
                "source_page":   page_num,
                "question_id":   f"{pdf_path.stem}_p{page_num}_q{n_soal:02d}",
                "extraction_method": "vlm",
            })
        elif typ == "materi":
            n_materi += 1
            records.append({
                "type": "materi",
                "judul":   (it.get("judul") or "").strip(),
                "content": (it.get("content") or "").strip(),
                "source_file":   pdf_path.name,
                "source_folder": pdf_path.parent.name,
                "subject":       subject,
                "source_page":   page_num,
                "question_id":   f"{pdf_path.stem}_p{page_num}_m{n_materi:02d}",
                "extraction_method": "vlm",
            })
    return records


def process_pdf(pdf_path: Path, max_pages: int | None = None, verbose: bool = True):
    pdf_path = Path(pdf_path)
    out_path = OUT_DIR / f"{pdf_path.parent.name}__{pdf_path.stem}.jsonl"
    if out_path.exists() and not OVERWRITE:
        if verbose:
            print(f"[skip] sudah ada: {out_path.name}")
        return out_path

    subject = _subject_for(pdf_path)
    all_records = []
    with fitz.open(pdf_path) as doc:
        n_pages = doc.page_count if max_pages is None else min(max_pages, doc.page_count)
        for pidx in range(n_pages):
            page = doc[pidx]
            img = page_to_image(page)
            raw = call_vlm_with_retry(img, EXTRACTION_PROMPT)
            items = parse_json_array(raw)
            recs = page_items_to_records(items, pdf_path, pidx + 1, subject)
            all_records.extend(recs)
            if verbose:
                ns = sum(r["type"] == "soal" for r in recs)
                nm = sum(r["type"] == "materi" for r in recs)
                print(f"  hal {pidx+1:>3}/{n_pages}: {ns} soal, {nm} materi")

    with open(out_path, "w", encoding="utf-8") as f:
        for r in all_records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    if verbose:
        ns = sum(r["type"] == "soal" for r in all_records)
        nm = sum(r["type"] == "materi" for r in all_records)
        print(f"[ok] {pdf_path.parent.name}/{pdf_path.name}: {ns} soal, {nm} materi -> {out_path.name}")
    return out_path


## 7. Daftar PDF & perkiraan skala

Lihat berapa PDF & total halaman sebelum batch (untuk perkiraan waktu/biaya).

In [ ]:
def list_pdfs(raw_dir: Path):
    return sorted(p for p in raw_dir.rglob("*.pdf"))

pdfs = list_pdfs(RAW_DIR)
total_pages = 0
print(f"{len(pdfs)} PDF ditemukan di {RAW_DIR}\n")
by_folder = {}
for p in pdfs:
    by_folder.setdefault(p.parent.name, 0)
    by_folder[p.parent.name] += 1
for folder, n in sorted(by_folder.items()):
    print(f"  {n:>3} PDF  {folder}")

# Hitung total halaman (cepat, hanya buka metadata)
for p in pdfs:
    try:
        with fitz.open(p) as d:
            total_pages += d.page_count
    except Exception as e:
        print(f"  ! gagal buka {p.name}: {e}")
print(f"\nTotal halaman (≈ jumlah request VLM): {total_pages}")


## 8. PROTOTYPE — uji di beberapa PDF dulu

Jalankan di sedikit PDF (campur ebook + kumpulan soal), batasi halaman, lalu
**periksa hasilnya** sebelum batch penuh. Sesuaikan `SAMPLE` & `MAX_PAGES`.

In [ ]:
# Pilih sampel: 1 ebook + beberapa kumpulan soal osn (sesuaikan nama folder bila perlu)
def pick_samples():
    picks = []
    # satu dari folder ebook/buku ajar mana pun selain osn
    ebook = next((p for p in pdfs if p.parent.name != "osn"), None)
    if ebook:
        picks.append(ebook)
    # dua dari osn
    picks += [p for p in pdfs if p.parent.name == "osn"][:2]
    return picks or pdfs[:3]

SAMPLE = pick_samples()
MAX_PAGES = 4   # batasi halaman per PDF saat prototype

print("Sampel:")
for p in SAMPLE:
    print("  ", p.relative_to(PROJECT_ROOT))
print()

_orig_overwrite = OVERWRITE
OVERWRITE = True  # selalu proses ulang saat prototype
for p in SAMPLE:
    process_pdf(p, max_pages=MAX_PAGES, verbose=True)
OVERWRITE = _orig_overwrite


### Periksa hasil prototype

Cetak beberapa record soal & materi untuk dicek kualitasnya.

In [ ]:
def preview_jsonl(stem_folder, n=6):
    path = OUT_DIR / f"{stem_folder}.jsonl"
    rows = [json.loads(l) for l in path.read_text(encoding="utf-8").splitlines()]
    print(f"== {path.name}: {len(rows)} record ==\n")
    for r in rows[:n]:
        if r["type"] == "soal":
            print(f"[SOAL hal {r['source_page']} no {r.get('nomor','')}] img={r['has_image']}")
            print("  Q:", r["question"][:300])
            if r["answer"]: print("  A:", r["answer"][:160])
            if r["steps"]:  print("  cara:", r["steps"][:200], "...")
        else:
            print(f"[MATERI hal {r['source_page']}] {r.get('judul','')}")
            print("  ", r["content"][:280])
        print("-" * 70)

for p in SAMPLE:
    preview_jsonl(f"{p.parent.name}__{p.stem}")
    print("\n")


## 9. BATCH PENUH

Setelah hasil prototype oke, proses semua PDF. Resumable: PDF yang outputnya
sudah ada akan di-skip (`OVERWRITE=False`). Aman dihentikan & dilanjutkan.

In [ ]:
from tqdm.auto import tqdm

errors = []
for p in tqdm(pdfs, desc="PDF"):
    try:
        process_pdf(p, max_pages=None, verbose=False)
    except Exception as e:
        errors.append((p, e))
        print(f"! error {p.name}: {e}")

print(f"\nSelesai. {len(pdfs)-len(errors)}/{len(pdfs)} PDF berhasil.")
if errors:
    print("Gagal:")
    for p, e in errors:
        print("  ", p.name, "->", e)


## 10. Gabung, pisah soal/materi, ringkasan

Gabungkan semua `.jsonl`, lalu tulis file terpisah untuk `soal` dan `materi`
plus CSV gabungan untuk inspeksi cepat.

In [ ]:
import csv

all_rows = []
for jf in sorted(OUT_DIR.glob("*.jsonl")):
    for line in jf.read_text(encoding="utf-8").splitlines():
        if line.strip():
            all_rows.append(json.loads(line))

soal   = [r for r in all_rows if r.get("type") == "soal"]
materi = [r for r in all_rows if r.get("type") == "materi"]

def write_jsonl(rows, name):
    path = OUT_DIR / name
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    return path

write_jsonl(soal,   "_all_soal.jsonl")
write_jsonl(materi, "_all_materi.jsonl")

# CSV gabungan (kolom union)
cols = ["type","subject","source_folder","source_file","source_page","question_id",
        "nomor","has_image","question","answer","steps","judul","content"]
csv_path = OUT_DIR / "_all.csv"
with open(csv_path, "w", encoding="utf-8", newline="") as f:
    w = csv.DictWriter(f, fieldnames=cols, extrasaction="ignore")
    w.writeheader()
    for r in all_rows:
        w.writerow(r)

print(f"Total record : {len(all_rows)}")
print(f"  soal       : {len(soal)}")
print(f"  materi     : {len(materi)}")
print(f"  soal+jawaban: {sum(1 for r in soal if r['answer'])}")
print(f"  soal+cara   : {sum(1 for r in soal if r['steps'])}")
print(f"  soal bergambar: {sum(1 for r in soal if r['has_image'])}")
print(f"\nDitulis:\n  {OUT_DIR/'_all_soal.jsonl'}\n  {OUT_DIR/'_all_materi.jsonl'}\n  {csv_path}")


## Langkah berikutnya

- `_all_soal.jsonl` siap masuk pipeline filter yang ada
  (`filter_rules` → `filter_validity` → `dedup`). Field sudah selaras.
- Mau pindah backend ke **Qwen2.5-VL** (lokal 4060 / Kaggle)? Isi `_qwen_call`
  lalu set `BACKEND="qwen"`. Sisa notebook tidak berubah.
- Mau **crop gambar diagram** ke file (bukan cuma deskripsi)? Bisa ditambah di
  `process_pdf` pakai `page.get_images()` / bbox PyMuPDF.
